# TFP-BD: Traffic congestion with ResNet50 and MobileNetV3-Large (frame classifier, 80/10/10)

Puro chobi dekhe model sorasori bole **Low / Medium / High**. Label = detector evaluation er **same niyom**:
annotation e gari gune (Pedestrian baad), 70/20/10 train set er 33 ar 66 percentile threshold (Low <= 7, Medium <= 11, High >= 12).
Test = **same 2,368 chobi** (sob location). Best epoch validation macro-F1 diye bachai, test e ekbar.
Shesh e 5 ta detector er sathe accuracy, macro-F1, bootstrap CI ar McNemar tulona; loss curve, accuracy curve, confusion matrix, classification report.

**Input (Add Input):**
1. Data Preparation (Sharmin) - `tfp_dataset`
2. **FRCNN_8020** er output (`split_8010_lists.json`)
3. **TFPBD_8020_Evaluation_All** er output (`dets_8020_all.pkl`, `compare_70_vs_80.csv`; 80/10/10 detector er sathe tulona) - Add Input -> Your Work -> Notebooks

**Settings:** GPU T4 x2 (ekta GPU use hobe), **Internet ON**. Cell 1 chalie dekho sob `OK`, tarpor **Save Version -> Save & Run All**.
Somoy anumanik 75-100 min. Shesh e Output e `cong_cls_8010_results.zip` -> amake pathao.

In [ ]:
import os, re, glob, time, math, random, zipfile, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

SPLIT_MODE = "80/10/10"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| split", SPLIT_MODE)

def safe(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")

WK = "/kaggle/working"
TAG = "7020" if SPLIT_MODE == "70/20/10" else "8010"
OUTD = f"{WK}/cong_cls_{TAG}_results"
os.makedirs(OUTD, exist_ok=True)

PED = {8}
LEVELS = ["Low", "Medium", "High"]
EXTS = (".jpg", ".jpeg", ".png", ".bmp")
DETECTORS = ["YOLOv8n", "YOLO11n", "RT-DETR-L", "Faster R-CNN", "EfficientDet-D3"]
CLS_MODELS = ["ResNet50", "MobileNetV3-Large"]
IMG_W, IMG_H = 448, 336
BATCH = 32
EPOCHS = {"ResNet50": 10, "MobileNetV3-Large": 10}
LR = {"ResNet50": 3e-4, "MobileNetV3-Large": 1e-3}
NUM_WORKERS = 4

F = {"SH": None, "SPLIT": None, "DET_CONG": None, "DET_PKL": None, "DET_CMP": None}
for root, dirs, files in os.walk("/kaggle/input"):
    if F["SH"] is None and "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            F["SH"] = c
    dirs[:] = [d for d in dirs if d not in ("tfp_dataset", "Bangladeshi Traffic Flow Dataset", "images", "labels", "tfp_8010")]
    for f in files:
        p = os.path.join(root, f)
        if f == "split_8010_lists.json":
            F["SPLIT"] = p
        elif f == "congestion_test_per_image.csv" and "eval_results_part2" in root:
            F["DET_CONG"] = p
        elif f == "dets_8020_all.pkl":
            F["DET_PKL"] = p
        elif f == "compare_70_vs_80.csv":
            F["DET_CMP"] = p
SH = F["SH"]
if SPLIT_MODE == "70/20/10":
    KEYS, need, DET_KEYS = ["SH", "DET_CONG"], {"SH"}, ["DET_CONG"]
    DET_SRC = "the Evaluation_part2 output"
else:
    KEYS, need, DET_KEYS = ["SH", "SPLIT", "DET_PKL", "DET_CMP"], {"SH", "SPLIT"}, ["DET_PKL", "DET_CMP"]
    DET_SRC = "the TFPBD_8020_Evaluation_All output"
print("\n" + "=" * 70)
for k in KEYS:
    v = F[k]
    tag = "OK " if v else ("MISSING" if k in need else "-- (optional)")
    print(f"{tag:14s} {k:9s} {v if v else ''}")
print("=" * 70)
assert SH, "tfp_dataset not found - add Sharmin's Data Preparation notebook with Add Input"
if SPLIT_MODE == "80/10/10":
    assert F["SPLIT"], "split_8010_lists.json not found - add the FRCNN_8020 notebook output with Add Input"
if not all(F[k] for k in DET_KEYS):
    print(f"Detector results not found: the comparison with the 5 detectors is skipped. Add {DET_SRC} to include it.")

try:
    torchvision.models.resnet50(weights="IMAGENET1K_V2")
    torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
    print("Pretrained weights OK (Internet ON)")
except Exception as e:
    raise RuntimeError("Could not download the pretrained weights - turn Internet ON in Settings. " + str(e)[:200])

In [ ]:
def stem_of(p):
    return os.path.splitext(os.path.basename(p))[0]

def boxes_from_label(lp):
    out = []
    if os.path.exists(lp):
        with open(lp) as fh:
            for line in fh:
                q = line.split()
                if len(q) >= 5:
                    c, xc, yc, w, h = int(q[0]), *map(float, q[1:5])
                    out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out

def n_vehicle(boxes):
    return sum(1 for c, b in boxes if c not in PED)

path_of, label_of, TEAM = {}, {}, {}
for s in ("train", "val", "test"):
    TEAM[s] = []
    for f in sorted(os.listdir(f"{SH}/{s}/images")):
        if f.lower().endswith(EXTS):
            st = stem_of(f)
            path_of[st] = f"{SH}/{s}/images/{f}"
            label_of[st] = f"{SH}/{s}/labels/{st}.txt"
            TEAM[s].append(st)
    TEAM[s] = sorted(TEAM[s])

train_counts = np.array([n_vehicle(boxes_from_label(label_of[st])) for st in TEAM["train"]])
Q33, Q66 = [float(x) for x in np.quantile(train_counts, [0.33, 0.66])]
def level(c):
    return 0 if c <= Q33 else (1 if c <= Q66 else 2)
print(f"Thresholds from the 70/20/10 train annotations: Low <= {Q33:g} | Medium <= {Q66:g} | High > {Q66:g}")
if (Q33, Q66) != (7.0, 11.0):
    print("Warning: thresholds differ from the detector evaluation (7 / 11)")

if SPLIT_MODE == "80/10/10":
    SPL = json.load(open(F["SPLIT"]))
    STEMS = {s: sorted(SPL[s]) for s in ("train", "val", "test")}
    assert STEMS["test"] == TEAM["test"], "80/10/10 test list differs from the 70/20/10 test set"
    missing = [st for s in ("train", "val") for st in STEMS[s] if st not in path_of]
    assert not missing, f"{len(missing)} split images not found in tfp_dataset, e.g. {missing[:3]}"
else:
    STEMS = TEAM

COUNT = {s: np.array([n_vehicle(boxes_from_label(label_of[st])) for st in STEMS[s]]) for s in ("train", "val", "test")}
LEVEL = {s: np.array([level(c) for c in COUNT[s]]) for s in ("train", "val", "test")}
for s in ("train", "val", "test"):
    d = pd.Series([LEVELS[x] for x in LEVEL[s]]).value_counts().reindex(LEVELS, fill_value=0).to_dict()
    print(f"{s:5s}: {len(STEMS[s]):6d} images | {d}")
assert len(STEMS["test"]) == 2368, "Test set should have 2,368 images"

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class FrameDS(Dataset):
    def __init__(self, split, train):
        self.items = [(path_of[st], int(y)) for st, y in zip(STEMS[split], LEVEL[split])]
        aug = [T.ColorJitter(0.3, 0.3, 0.2),
               T.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.95, 1.05))] if train else []
        self.tf = T.Compose([T.Resize((IMG_H, IMG_W))] + aug + [T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def make_loader(split, train):
    return DataLoader(FrameDS(split, train), batch_size=BATCH, shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=DEV.type == "cuda", persistent_workers=NUM_WORKERS > 0, drop_last=train)

LOADERS = {s: make_loader(s, s == "train") for s in ("train", "val", "test")}

def build_model(name):
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, 3)
    elif name == "MobileNetV3-Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 3)
    else:
        raise ValueError(name)
    return m.to(DEV)

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=DEV.type == "cuda")

@torch.no_grad()
def predict(model, split, crit=None):
    model.eval()
    probs, loss_sum, n = [], 0.0, 0
    for x, y in LOADERS[split]:
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
            logits = model(x.to(DEV, non_blocking=True))
        if crit is not None:
            loss_sum += crit(logits.float(), y.to(DEV)).item() * len(y)
            n += len(y)
        probs.append(torch.softmax(logits.float(), 1).cpu().numpy())
    prob = np.concatenate(probs)
    return prob.argmax(1), prob, (loss_sum / max(n, 1) if crit is not None else None)

def train_model(name):
    model = build_model(name)
    epochs, steps = EPOCHS[name], len(LOADERS["train"])
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR[name], total_steps=epochs * steps, pct_start=0.15)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = make_scaler()
    best, hist, t0 = -1.0, [], time.time()
    ckpt = f"{WK}/cong_{safe(name)}_{TAG}_best.pt"
    gv = LEVEL["val"]
    for ep in range(1, epochs + 1):
        model.train()
        run_loss, correct, n = 0.0, 0, 0
        for x, y in LOADERS["train"]:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * len(y)
            correct += int((logits.argmax(1) == y).sum().item())
            n += len(y)
        pv, _, val_loss = predict(model, "val", crit)
        vf1 = f1_score(gv, pv, average="macro") * 100
        hist.append({"epoch": ep, "train_loss": run_loss / n, "train_acc_%": correct / n * 100,
                     "val_loss": val_loss, "val_acc_%": float((pv == gv).mean() * 100),
                     "val_macro_f1_%": vf1, "minutes": (time.time() - t0) / 60})
        tag = ""
        if vf1 > best:
            best = vf1
            torch.save(model.state_dict(), ckpt)
            tag = "  <- best"
        h = hist[-1]
        print(f"[{name}] epoch {ep}/{epochs} | train loss {h['train_loss']:.4f} acc {h['train_acc_%']:.1f}% | "
              f"val loss {h['val_loss']:.4f} acc {h['val_acc_%']:.1f}% macro-F1 {vf1:.1f}% | {h['minutes']:.1f} min{tag}")
    pd.DataFrame(hist).to_csv(f"{OUTD}/history_cong_{safe(name)}.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEV))
    return model, hist, (time.time() - t0) / 60

xb, yb = next(iter(LOADERS["val"]))
print("Batch:", tuple(xb.shape), "| labels:", yb[:16].tolist())
for name in CLS_MODELS:
    m = build_model(name)
    m.train()
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
        out = m(xb[:4].to(DEV))
    out.float().sum().backward()
    print(f"{name}: forward and backward OK, output {tuple(out.shape)}")
    del m, out
if DEV.type == "cuda":
    torch.cuda.empty_cache()
print("Functions ready")

In [ ]:
MODELS = {}
MODELS["ResNet50"] = train_model("ResNet50")

In [ ]:
MODELS["MobileNetV3-Large"] = train_model("MobileNetV3-Large")

In [ ]:
@torch.no_grad()
def speed_ms(model, n=200):
    model.eval()
    ds = FrameDS("test", False)
    xs = [ds[i][0].unsqueeze(0).to(DEV) for i in range(min(n, len(ds)))]
    for x in xs[:20]:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for x in xs:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t) / len(xs) * 1000

g = LEVEL["test"]
tab = pd.DataFrame({"stem": STEMS["test"], "gt_count": COUNT["test"], "gt_level": g})
rows = []
for name, (model, hist, train_min) in MODELS.items():
    pred, prob, _ = predict(model, "test")
    tab[name] = pred
    for k in range(3):
        tab[f"{name}_p_{LEVELS[k]}"] = prob[:, k]
    rows.append({"model": name, "n": len(pred), "accuracy_%": (pred == g).mean() * 100,
                 "macro_f1_%": f1_score(g, pred, average="macro") * 100,
                 "kappa_weighted": cohen_kappa_score(g, pred, weights="linear"),
                 "far_miss_(Low<->High)": int(((pred == 0) & (g == 2)).sum() + ((pred == 2) & (g == 0)).sum()),
                 "params_M": round(sum(q.numel() for q in model.parameters()) / 1e6, 2),
                 "ms_per_image_bs1": round(speed_ms(model), 2), "train_minutes": round(train_min, 1),
                 "best_epoch": max(hist, key=lambda h: h["val_macro_f1_%"])["epoch"]})
cls_df = pd.DataFrame(rows)
cls_df.to_csv(f"{OUTD}/congestion_test_classifiers.csv", index=False)
print("=" * 70 + f"\nCONGESTION, FRAME CLASSIFIERS ({SPLIT_MODE}) | test {len(tab)} images\n" + "=" * 70)
print(cls_df.round(3).to_string(index=False))

NAMES = list(MODELS)
if SPLIT_MODE == "70/20/10" and F["DET_CONG"]:
    det = pd.read_csv(F["DET_CONG"]).sort_values("stem").reset_index(drop=True)
    det_gt = det["gt_level"].map(LEVELS.index).values
    same = len(det) == len(tab) and (det["stem"].astype(str).values == tab["stem"].values).all() and (det_gt == g).all()
    if same:
        for c in DETECTORS:
            tab[c] = det[f"{c}_level"].map(LEVELS.index).values
        NAMES = DETECTORS + NAMES
        print("\nDetector results matched image by image (same stems, same ground-truth level)")
    else:
        print("\nDetector file does not match these test images: comparison skipped")
elif SPLIT_MODE == "80/10/10" and F["DET_PKL"] and F["DET_CMP"]:
    import pickle
    D8 = pickle.load(open(F["DET_PKL"], "rb"))["NEW_TEST"]
    cmp8 = pd.read_csv(F["DET_CMP"])
    cmp8 = cmp8[cmp8["split"] == "80/10/10"].set_index("model")
    chk = []
    for c in DETECTORS:
        thr = float(cmp8.loc[c, "cong_thr"])
        cnt = np.array([sum(1 for k, b, sc in D8[c].get(st, []) if sc >= thr and k not in PED) for st in tab["stem"]])
        tab[c] = np.array([level(x) for x in cnt])
        chk.append({"model": c, "thr": thr, "accuracy_here_%": (tab[c].to_numpy() == g).mean() * 100,
                    "accuracy_Evaluation_All_%": float(cmp8.loc[c, "cong_acc_%"])})
    chk = pd.DataFrame(chk)
    print("\n80/10/10 detectors rebuilt from dets_8020_all.pkl (check against Evaluation_All):")
    print(chk.round(3).to_string(index=False))
    if (chk["accuracy_here_%"] - chk["accuracy_Evaluation_All_%"]).abs().max() < 0.05:
        NAMES = DETECTORS + NAMES
        print("Match OK")
    else:
        for c in DETECTORS:
            del tab[c]
        print("Mismatch with Evaluation_All: detector comparison skipped")
tab.to_csv(f"{OUTD}/congestion_test_per_image_classifiers.csv", index=False)

rng = np.random.default_rng(42)
_ = rng.integers(0, 1605, (1000, 1605))
ic = rng.integers(0, len(tab), (1000, len(tab)))
all_rows = []
for name in NAMES:
    p = tab[name].to_numpy()
    acc = [(p[ix] == g[ix]).mean() * 100 for ix in ic]
    f1s = [f1_score(g[ix], p[ix], average="macro") * 100 for ix in ic[:300]]
    all_rows.append({"model": name, "type": "classifier" if name in MODELS else "detector",
                     "accuracy_%": (p == g).mean() * 100, "acc_CI_low": np.percentile(acc, 2.5),
                     "acc_CI_high": np.percentile(acc, 97.5), "macro_f1_%": f1_score(g, p, average="macro") * 100,
                     "f1_CI_low": np.percentile(f1s, 2.5), "f1_CI_high": np.percentile(f1s, 97.5),
                     "kappa_weighted": cohen_kappa_score(g, p, weights="linear"),
                     "far_miss": int(((p == 0) & (g == 2)).sum() + ((p == 2) & (g == 0)).sum())})
all_df = pd.DataFrame(all_rows)
all_df.to_csv(f"{OUTD}/congestion_test_all_models.csv", index=False)
print("\nAll models on the same test images (accuracy with bootstrap 95% CI, 1000 resamples, seed 42):")
print(all_df.round(3).to_string(index=False))

mc = []
for i in range(len(NAMES)):
    for j in range(i + 1, len(NAMES)):
        a = tab[NAMES[i]].to_numpy() == g
        b = tab[NAMES[j]].to_numpy() == g
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        pv = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        mc.append({"model_A": NAMES[i], "model_B": NAMES[j], "only_A_correct": n01, "only_B_correct": n10,
                   "p_value": pv, "significant_(p<0.05)": pv < 0.05})
mc_df = pd.DataFrame(mc)
mc_df.to_csv(f"{OUTD}/congestion_mcnemar_all_models.csv", index=False)
print("\nMcNemar (pairs with at least one classifier):")
print(mc_df[mc_df["model_A"].isin(MODELS) | mc_df["model_B"].isin(MODELS)].round(4).to_string(index=False))

rep_rows = []
for name in NAMES:
    p = tab[name].to_numpy()
    rep = classification_report(g, p, labels=[0, 1, 2], target_names=LEVELS, digits=4, output_dict=True, zero_division=0)
    for k, v in rep.items():
        rep_rows.append({"model": name, "row": k, **v} if isinstance(v, dict) else {"model": name, "row": k, "f1-score": v})
    if name in MODELS:
        print(f"\nClassification report, {name} (test, {len(tab)} images):")
        print(classification_report(g, p, labels=[0, 1, 2], target_names=LEVELS, digits=4, zero_division=0))
pd.DataFrame(rep_rows).to_csv(f"{OUTD}/classification_report_all_models.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for name, (model, hist, _) in MODELS.items():
    h = pd.DataFrame(hist)
    col = ax[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"{name} train")[0].get_color()
    ax[0].plot(h["epoch"], h["val_loss"], marker="s", ls="--", color=col, label=f"{name} validation")
    ax[1].plot(h["epoch"], h["train_acc_%"], marker="o", color=col, label=f"{name} train")
    ax[1].plot(h["epoch"], h["val_acc_%"], marker="s", ls="--", color=col, label=f"{name} validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Cross-entropy loss"); ax[0].set_title("Loss curve"); ax[0].legend()
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy (%)"); ax[1].set_title("Accuracy curve"); ax[1].legend()
plt.tight_layout()
plt.savefig(f"{OUTD}/cong_classifiers_loss_accuracy_curves.png", dpi=110, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, len(MODELS), figsize=(6.5 * len(MODELS), 5.5))
for a, name in zip(np.atleast_1d(ax), MODELS):
    cm = confusion_matrix(g, tab[name].to_numpy(), labels=[0, 1, 2])
    a.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            a.text(j, i, f"{cm[i, j]}\n{cm[i, j] / max(cm[i].sum(), 1) * 100:.1f}%", ha="center", va="center", fontsize=12,
                   color="white" if cm[i, j] > cm.max() / 2 else "black")
    a.set_xticks(range(3)); a.set_xticklabels(LEVELS); a.set_xlabel("Model prediction")
    a.set_yticks(range(3)); a.set_yticklabels(LEVELS); a.set_ylabel("Annotation")
    a.set_title(f"{name} (test)")
plt.tight_layout()
plt.savefig(f"{OUTD}/cong_classifiers_confusion_matrices.png", dpi=110, bbox_inches="tight")
plt.show()

o = all_df.sort_values("accuracy_%")
fig, ax = plt.subplots(figsize=(9, 0.6 * len(o) + 1.5))
ax.errorbar(o["accuracy_%"], range(len(o)), xerr=[o["accuracy_%"] - o["acc_CI_low"], o["acc_CI_high"] - o["accuracy_%"]],
            fmt="o", capsize=4)
ax.set_yticks(range(len(o))); ax.set_yticklabels(o["model"])
ax.set_xlabel("Congestion accuracy on test (%), 95% CI"); ax.set_title(f"Congestion, {len(tab)} test images ({SPLIT_MODE})")
plt.tight_layout()
plt.savefig(f"{OUTD}/cong_all_models_accuracy_ci.png", dpi=110, bbox_inches="tight")
plt.show()

zp = f"{WK}/cong_cls_{TAG}_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTD)):
        z.write(os.path.join(OUTD, f), f)
print("\nSaved:", zp)